In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
os.environ["LANGSMITH_TRACING"] = "true"
openai_api_key = os.environ["OPENAI_API_KEY"]
langsmith_api_key = os.environ["LANGSMITH_API_KEY"]
print(f"openai_api_key: f{openai_api_key}")
print(f"langsmith_api_key: f{langsmith_api_key}")

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

In [ ]:
docs = [WebBaseLoader(url).load() for url in urls]

In [ ]:
docs

In [ ]:
docs_list = [item for sublist in docs for item in sublist]
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)

In [ ]:
doc_splits

In [ ]:
vectorstore = InMemoryVectorStore.from_documents(documents=doc_splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(k=6)

In [ ]:
retriever.invoke("What is agents")

In [ ]:
chat_llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langsmith import traceable


@traceable()
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)
    instructions = f"""
    You are a helpful assistant who is good at analyzing source information and answering questions.
    Use the following source documents to answer the user's questions.       
    If you don't know the answer, just say that you don't know.       
    Use three sentences maximum and keep the answer concise.
    Documents:
    {docs_string}
    """
    
    ai_msg = chat_llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    return {"answer": ai_msg.content, "documents": docs}
    

In [ ]:
rag_bot("What is agents")

In [ ]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."
        },
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {
            "answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."
        },
    },
]

dataset_name = "@saranshkhulbe7/rag-test-evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

In [ ]:
from typing_extensions import Annotated, TypedDict


class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
    QUESTION: {inputs["question"]}
    GROUND TRUTH ANSWER: {reference_outputs["answer"]}
    STUDENT ANSWER: {outputs["answer"]}"""

    grade = grader_llm.invoke(
        [
            {"role": "system", "content": correctness_instructions},
            {"role": "user", "content": answers},
        ]
    )
    return grade["correct"]

In [ ]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool, ..., "Provide the score on whether the answer addresses the question"
    ]


relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)


def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke(
        [
            {"role": "system", "content": relevance_instructions},
            {"role": "user", "content": answer},
        ]
    )
    return grade["relevant"]


In [ ]:
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[
        bool, ..., "Provide the score on if the answer hallucinates from the documents"
    ]


grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)


def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke(
        [
            {"role": "system", "content": grounded_instructions},
            {"role": "user", "content": answer},
        ]
    )
    return grade["grounded"]


In [ ]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise",
    ]


retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

retrieval_relevance_llm = ChatOpenAI(
    model="gpt-4o", temperature=0
).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)


def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    grade = retrieval_relevance_llm.invoke(
        [
            {"role": "system", "content": retrieval_relevance_instructions},
            {"role": "user", "content": answer},
        ]
    )
    return grade["relevant"]


In [33]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, relevance, groundedness, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)

/Users/saranshkhulbe/Documents/Projects/harkirat/python-for-ai/deep-agents-proj/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'rag-doc-relevance-cdb64556' at:
https://smith.langchain.com/o/57f93a6b-820f-42eb-b88e-94a9ab3132d6/datasets/835e8d55-62c6-44f1-ae01-6bc03e7d7030/compare?selectedSessions=853adb9d-fc57-4811-ab0b-ed3b5c428d3d




3it [00:29,  9.88s/it]


In [36]:
experiment_results

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.relevance,feedback.groundedness,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,The types of biases that can arise with few-sh...,[page_content='Zero-shot and few-shot learning...,None,The biases that can arise with few-shot prompt...,True,True,True,True,4.149130,1cd563dc-8965-45ee-b219-0b7a05ff348a,01a00c54-eb84-70a0-8051-fd47273d31ed
1,What are five types of adversarial attacks?,The five types of adversarial attacks are toke...,[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,2.197223,6d8e39ac-8a4b-4896-9e75-a23e03e6ba0f,01a00c55-1654-7a93-b9b9-768e1978b3b5
2,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection to refine...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,False,False,1.746845,78a58aaa-a99b-4a0a-8c3b-494ac23eac97,01a00c55-353c-77a2-b1e0-e9cb8a5aa391


In [35]:
experiment_results.to_pandas()

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.relevance,feedback.groundedness,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,The types of biases that can arise with few-sh...,[page_content='Zero-shot and few-shot learning...,None,The biases that can arise with few-shot prompt...,True,True,True,True,4.149130,1cd563dc-8965-45ee-b219-0b7a05ff348a,01a00c54-eb84-70a0-8051-fd47273d31ed
1,What are five types of adversarial attacks?,The five types of adversarial attacks are toke...,[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,2.197223,6d8e39ac-8a4b-4896-9e75-a23e03e6ba0f,01a00c55-1654-7a93-b9b9-768e1978b3b5
2,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection to refine...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,False,False,1.746845,78a58aaa-a99b-4a0a-8c3b-494ac23eac97,01a00c55-353c-77a2-b1e0-e9cb8a5aa391
